In [1]:
from monai.networks.nets import UNet
from monai.networks.layers import Norm
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

2026-01-21 03:01:16.538836: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-21 03:01:17.934416: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


cuda


In [2]:
model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=1,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
    norm=Norm.BATCH
).to(device)

print(model)

model.load_state_dict(torch.load("best_metric_model_new.pth"))

UNet(
  (model): Sequential(
    (0): ResidualUnit(
      (conv): Sequential(
        (unit0): Convolution(
          (conv): Conv3d(1, 16, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1))
          (adn): ADN(
            (N): BatchNorm3d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (D): Dropout(p=0.0, inplace=False)
            (A): PReLU(num_parameters=1)
          )
        )
        (unit1): Convolution(
          (conv): Conv3d(16, 16, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
          (adn): ADN(
            (N): BatchNorm3d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (D): Dropout(p=0.0, inplace=False)
            (A): PReLU(num_parameters=1)
          )
        )
      )
      (residual): Conv3d(1, 16, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1))
    )
    (1): SkipConnection(
      (submodule): Sequential(
        (0): ResidualUnit(
          (conv): Sequential(


<All keys matched successfully>

In [3]:
import dataloading.load_atlas as load_atlas

train_id_loader, val_id_loader, test_id_loader = load_atlas.load_atlas()

Found 655 image files
Found 655 mask files
Found 300 test image files
Train data: 524 files
Validation data: 131 files


monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.


In [4]:
from monai.inferers import sliding_window_inference
from tqdm import tqdm
import pickle
from pathlib import Path

def compute_id_scores(model, id_loader):
    model.eval()
    id_scores = []
    with torch.no_grad():
        for batch, batch_data in enumerate(tqdm(id_loader, "Computing ID scores")):
            image = batch_data["image"].to(device)
            if image.ravel().shape[0] == 0:
                print("Empty image encountered, skipping. Batch:", batch)
                continue
            output = sliding_window_inference(image, (96, 96, 96), 4, model, overlap=0.5)
            confidence_score = torch.sigmoid(output)
            mcs = torch.max(confidence_score, 1 - confidence_score)
            id_scores.append(torch.min(mcs).cpu().item())
    return id_scores

pickle_filename = "id_scores.pkl"
if Path(pickle_filename).exists():
    with open(pickle_filename, "rb") as fp:
        id_scores = pickle.load(fp)
else:
    id_scores = compute_id_scores(model, val_id_loader)
    with open(pickle_filename, "wb") as fp:
        pickle.dump(id_scores, fp)

id_scores


[0.5000205636024475,
 0.5000014305114746,
 0.5000021457672119,
 0.5000171661376953,
 0.5000507831573486,
 0.5000019073486328,
 0.500002384185791,
 0.5000447034835815,
 0.5000441074371338,
 0.5000371932983398,
 0.5000326633453369,
 0.5000009536743164,
 0.5000190734863281,
 0.5000443458557129,
 0.500054121017456,
 0.5000206828117371,
 0.5000169277191162,
 0.5000190734863281,
 0.5000521540641785,
 0.5000095367431641,
 0.5000460743904114,
 0.5000075101852417,
 0.5000132322311401,
 0.5000081062316895,
 0.50007164478302,
 0.5000400543212891,
 0.5000066757202148,
 0.5001029968261719,
 0.500030517578125,
 0.5000033378601074,
 0.5000401735305786,
 0.500004768371582,
 0.5000591278076172,
 0.5000782012939453,
 0.5000220537185669,
 0.5000643134117126,
 0.5000038146972656,
 0.5000298619270325,
 0.5000522136688232,
 0.5000009536743164,
 0.5000276565551758,
 0.500044584274292,
 0.5000095367431641,
 0.5000219345092773,
 0.5000200271606445,
 0.5000026822090149,
 0.5000019073486328,
 0.5000039339065552,

In [5]:
val_scores = compute_id_scores(model, val_id_loader)

Computing ID scores:   0%|          | 0/131 [00:00<?, ?it/s]Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /build/python-pytorch/src/pytorch-cuda/torch/csrc/autograd/python_variable_indexing.cpp:345.)
Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /build/python-pytorch/src/pytorch-cuda/torch/csrc/autograd/python_variable_indexing.cpp:345.)
Computing ID scores:  60%|█████▉    | 78/131 [00:29<00:20,  2.63it/s]


KeyboardInterrupt: 

In [6]:
import numpy as np

def determine_threshold(id_scores, percentile=5):
    threshold = np.percentile(id_scores, percentile)
    return threshold

threshold = determine_threshold(val_scores, percentile=5)

In [ ]:
from dataloading.load_chaos import load_chaos

chaos_loader = load_chaos()

Found 20 Train CT volumes
Found 20 Test CT volumes
Found 0 Train MR volumes
Found 0 Test MR volumes
Total CHAOS volumes: 40
CHAOS dataloader created with 40 samples


In [ ]:
id_scores_chaos = compute_id_scores(model, chaos_loader)
id_scores_chaos

Computing ID scores: 100%|██████████| 40/40 [01:04<00:00,  1.62s/it]


[0.5000036954879761,
 0.5000114440917969,
 0.500000536441803,
 0.5000004768371582,
 0.5000038146972656,
 0.5000035166740417,
 0.5000135898590088,
 0.5000014901161194,
 0.5000247955322266,
 0.5000035166740417,
 0.5000038743019104,
 0.5000271797180176,
 0.5000119209289551,
 0.5000128746032715,
 0.5000033378601074,
 0.5000009536743164,
 0.5000009536743164,
 0.5000085830688477,
 0.5000791549682617,
 0.5000036358833313,
 0.5000004172325134,
 0.5000150203704834,
 0.5,
 0.5000131130218506,
 0.5000009536743164,
 0.5000317096710205,
 0.5000028610229492,
 0.5000019073486328,
 0.5000009536743164,
 0.5000048875808716,
 0.500002384185791,
 0.5000038146972656,
 0.5000028610229492,
 0.5000044703483582,
 0.5000038146972656,
 0.5000013709068298,
 0.5000054240226746,
 0.5000014305114746,
 0.5000147819519043,
 0.5000033378601074]

In [ ]:
test_id_scores = np.array(compute_id_scores(model, test_id_loader))
id_scores = np.concatenate([np.array(id_scores_chaos), test_id_scores])
labels = np.concatenate([np.ones_like(id_scores_chaos), np.zeros_like(test_id_scores)])

Computing ID scores: 100%|██████████| 300/300 [01:57<00:00,  2.56it/s]


In [ ]:
from sklearn.metrics import average_precision_score
from metrics import fpr

anomaly_scores = 1-id_scores

aupr = average_precision_score(labels, anomaly_scores)
fpr90 = fpr(labels, anomaly_scores, percentile=10)
fpr95 = fpr(labels, anomaly_scores, percentile=5)
fpr99 = fpr(labels, anomaly_scores, percentile=1)

print(f"AUPR: {aupr:.4f}, fpr90: {fpr90:.4f}, fpr95: {fpr95:.4f}, fpr99: {fpr99:.4f}")

AUPR: 0.2314, fpr90: 0.5967, fpr95: 0.7733, fpr99: 0.9767


In [6]:
from dataloading.load_brats import load_brats
brats_loader = load_brats()
len(brats_loader)

Found 585 Train BRATS volumes
Found 87 Test BRATS volumes


672

In [7]:
id_scores_brats = compute_id_scores(model, brats_loader)
id_scores_brats

Computing ID scores:   0%|          | 0/672 [00:00<?, ?it/s]WARNING: In /work/ITK-source/ITK/Modules/IO/ImageBase/include/itkImageSeriesReader.hxx, line 478
ImageSeriesReader (0x56424252ab20): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.00101728

ImageSeriesReader (0x56424252ab20): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000545123

Computing ID scores:   0%|          | 3/672 [00:05<15:20,  1.38s/it]WARNING: In /work/ITK-source/ITK/Modules/IO/ImageBase/include/itkImageSeriesReader.hxx, line 478
ImageSeriesReader (0x56424252ab20): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.00100171

Computing ID scores:   4%|▎         | 25/672 [00:17<02:16,  4.74it/s]WARNING: In /work/ITK-source/ITK/Modules/IO/ImageBase/include/itkImageSeriesReader.hxx, line 478
ImageSeriesReader (0x56424252ab20): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000668705

ImageSeriesReader (0x56424252a

Empty image encountered, skipping. Batch: 81


Computing ID scores:  43%|████▎     | 286/672 [01:50<01:49,  3.53it/s]WARNING: In /work/ITK-source/ITK/Modules/IO/ImageBase/include/itkImageSeriesReader.hxx, line 478
ImageSeriesReader (0x56424252ab20): Non uniform sampling or missing slices detected,  maximum nonuniformity:9.95169e-05

Computing ID scores:  44%|████▍     | 299/672 [01:53<01:11,  5.24it/s]WARNING: In /work/ITK-source/ITK/Modules/IO/ImageBase/include/itkImageSeriesReader.hxx, line 478
ImageSeriesReader (0x56424252ab20): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000136793

ImageSeriesReader (0x56424252ab20): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000398104

Computing ID scores:  45%|████▌     | 303/672 [01:55<01:38,  3.74it/s]WARNING: In /work/ITK-source/ITK/Modules/IO/ImageBase/include/itkImageSeriesReader.hxx, line 478
ImageSeriesReader (0x56424252ab20): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000199052

ImageSeriesRead

[0.5000925064086914,
 0.5010952949523926,
 0.5000767707824707,
 0.5181318521499634,
 0.5002707839012146,
 0.5000014305114746,
 0.9997778534889221,
 0.5000879764556885,
 0.5000829696655273,
 0.9995916485786438,
 0.5000686645507812,
 0.5001654624938965,
 0.5002121925354004,
 0.5000724792480469,
 0.5001068115234375,
 0.5000848770141602,
 0.5000267028808594,
 0.500385582447052,
 0.5007253885269165,
 0.5000738501548767,
 0.500062108039856,
 0.5002583265304565,
 0.5000252723693848,
 0.5007874965667725,
 0.5001888275146484,
 0.5003671646118164,
 0.5000996589660645,
 0.5000505447387695,
 0.5002131462097168,
 0.5000133514404297,
 0.5007498264312744,
 0.9999294877052307,
 0.5000240802764893,
 0.5000219345092773,
 0.5002102851867676,
 0.5000145435333252,
 0.5002739429473877,
 0.5008525848388672,
 0.5002379417419434,
 0.5001249313354492,
 0.5000748634338379,
 0.5001192092895508,
 0.5002081394195557,
 0.5166316628456116,
 0.5002956390380859,
 0.5000464916229248,
 0.5000138282775879,
 0.500058174133

In [9]:
import numpy as np

test_id_scores = np.array(compute_id_scores(model, test_id_loader))
id_scores = np.concatenate([np.array(id_scores_brats), test_id_scores])
labels = np.concatenate([np.ones_like(id_scores_brats), np.zeros_like(test_id_scores)])

Computing ID scores:   0%|          | 0/300 [00:00<?, ?it/s]Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /build/python-pytorch/src/pytorch-cuda/torch/csrc/autograd/python_variable_indexing.cpp:345.)
Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /build/python-pytorch/src/pytorch-cuda/torch/csrc/autograd/python_variable_indexing.cpp:345.)
Computing ID scores: 100%|██████████| 300/300 [01:50<00:00,  2.72it/s]


In [10]:
from sklearn.metrics import average_precision_score
from metrics import fpr

anomaly_scores = -id_scores

aupr = average_precision_score(labels, anomaly_scores)
fpr90 = fpr(labels, anomaly_scores, percentile=10)
fpr95 = fpr(labels, anomaly_scores, percentile=5)
fpr99 = fpr(labels, anomaly_scores, percentile=1)

print(f"AUPR: {aupr:.4f}, fpr90: {fpr90:.4f}, fpr95: {fpr95:.4f}, fpr99: {fpr99:.4f}")

AUPR: 0.5007, fpr90: 1.0000, fpr95: 1.0000, fpr99: 1.0000
